# Missing-address matcher retraining experiment
Reuse the full training combined-vector cache, the 30k/2k/2k saved split, and the completed name-only missing-address pool. Attach **ml-needed**, **full_pool_embedding_v1**, and the saved **name_only_rescue_v1** notebook output. Enable a T4 GPU and Internet for pinned public E5 weights. Only supplied business records are used.

Candidate policy: combined search (256 partitions, 200 initial neighbors, top20) union same-country name-only top5 from ALL blank-address S2/S3 records. No true pairs are injected. Every model is compared on the same union candidate set. A baseline-only prediction row is also retained for comparison.

Models: saved matcher; retrained baseline-pair control (19 features); retrained union pairs (19 features); retrained union pairs (20 features, adding name cosine). Same HGB hyperparameters and same training businesses. No class weighting or automatic acceptance rules. Thresholds are selected only on 2,000 tuning businesses. All 2,000 evaluation businesses are kept out of training and threshold selection, but this evaluation split has been inspected in earlier experiments; it is **not a pristine final test**.

This notebook generates some NEW name embeddings for training queries and retrieved records with addresses. It reuses the saved 344k blank-address name embeddings and ALL combined-text embeddings. It saves checkpoints and experimental models; it does not generate a competition submission or automatically promote a model.


In [ ]:
%pip install -q numpy==2.0.2 pandas==2.3.3 scikit-learn==1.6.1 faiss-cpu==1.11.0 threadpoolctl transformers==5.0.0 safetensors sentencepiece


In [ ]:
from pathlib import Path
import os,sys,json,pickle,shutil,gc,time,hashlib,sqlite3,zipfile
import numpy as np,pandas as pd
from threadpoolctl import threadpool_limits
import torch,faiss
from sklearn.ensemble import HistGradientBoostingClassifier
assert torch.cuda.is_available(), 'Enable GPU before running.'
os.environ['TOKENIZERS_PARALLELISM']='false'
UPGRADE_PATH=''
CACHE_PATH=''
NAME_RUN_PATH=''  # Folder containing configuration.json and name_pool/records.pkl
ROOT=Path('/kaggle/working/matcher_name_rescue_v1')
OUT=ROOT/'results';OUT.mkdir(parents=True,exist_ok=True)
BATCH_SIZE=64
PAIR_CHUNK=500

def discover(explicit,marker,required):
    paths=[Path(explicit)] if explicit else sorted({p.parent for p in Path('/kaggle/input').rglob(marker)})
    paths=[p for p in paths if all((p/n).exists() for n in required)]
    assert len(paths)==1, f'Set the explicit path; found {paths}'
    return paths[0]
UPGRADE=discover(UPGRADE_PATH,'upgrade_report.json',['sample.pkl','queries.npy','model/matcher.pkl','model/experiment_results.json'])
SOURCE=discover(CACHE_PATH,'encoding_complete.json',['vectors.f16','records.sqlite','index_catalog.json','sample.tsv','configuration.json'])
if NAME_RUN_PATH: name_runs=[Path(NAME_RUN_PATH)]
else: name_runs=sorted({p.parent.parent for p in Path('/kaggle/input').rglob('records.pkl') if p.parent.name=='name_pool'})
name_runs=[p for p in name_runs if (p/'name_pool/embedding_parts').is_dir() and (p/'configuration.json').exists()]
assert len(name_runs)==1,f'Set NAME_RUN_PATH to the saved name_only_rescue_v1 folder; found {name_runs}'
NAME_RUN=name_runs[0]
SAVED_CODE=UPGRADE.parent/'code'
assert (SAVED_CODE/'submit_pipeline.py').exists()
sys.path.insert(0,str(SAVED_CODE))
import embedding_core as c, submit_pipeline as s
old_bundle,old_meta=s.load_model(UPGRADE/'model')
with (UPGRADE/'sample.pkl').open('rb') as f:sample,truth=pickle.load(f)
q=np.load(UPGRADE/'queries.npy',mmap_mode='r')
assert sample.entity_id.is_unique and set(sample.split)=={'train','tune','evaluation'}
assert len(q)==len(sample) and q.shape[1]==c.DIM
print('Split:',sample.split.value_counts().to_dict())
print('Upgrade:',UPGRADE,'\nCombined cache:',SOURCE,'\nSaved name pool:',NAME_RUN)
name_config=json.loads((NAME_RUN/'configuration.json').read_text())
assert name_config['revision']==c.REVISION and name_config['model']==c.MODEL_ID
assert name_config['text']=='query: NAME' and name_config['max_length']==256
config={'version':'matcher-name-rescue-v1','revision':c.REVISION,'batch_size':BATCH_SIZE,'pair_chunk':PAIR_CHUNK,
        'nprobe':256,'overfetch':200,'baseline_k':20,'name_k':5,'threshold_grid':'0.01:0.99:0.01',
        'sample_hash':s.sha256(UPGRADE/'sample.pkl'),'query_hash':s.sha256(UPGRADE/'queries.npy'),
        'old_model_hash':s.sha256(UPGRADE/'model/matcher.pkl'),'core_hash':s.sha256(SAVED_CODE/'embedding_core.py'),
        'runner_hash':s.sha256(SAVED_CODE/'submit_pipeline.py'),
        'combined_config_hash':s.sha256(SOURCE/'configuration.json'),
        'name_pool_hash':s.sha256(NAME_RUN/'name_pool/records.pkl'),
        'name_config':name_config,'source_cache':str(SOURCE),'upgrade':str(UPGRADE),'name_run':str(NAME_RUN)}
cp=ROOT/'configuration.json'
if cp.exists():assert json.loads(cp.read_text())==config,'Configuration changed; use a new ROOT.'
else:c.atomic_json(cp,config)


In [ ]:
# Copy immutable cache files locally; never alter the attached source.
prior_caches = [Path('/kaggle/working/name_only_rescue_v1/local_cache'),Path('/kaggle/working/entity_gap_audit_v1/local_cache'),Path('/kaggle/working/faiss_search_ablation_v1/local_cache')]
CACHE = next((p for p in prior_caches if (p/'copy_identity.json').exists()), ROOT/'local_cache')
CACHE.mkdir(exist_ok=True)
catalog = json.loads((SOURCE/'index_catalog.json').read_text())
names = ['vectors.f16','records.sqlite','encoding_complete.json','index_catalog.json'] + [x['file'] for x in catalog]
manifest = {'source':str(SOURCE), 'sizes':{n:(SOURCE/n).stat().st_size for n in names}}
identity = CACHE/'copy_identity.json'
if identity.exists():
    assert json.loads(identity.read_text()) == manifest, 'Different source cache; use a new ROOT.'
else:
    assert not any(CACHE.iterdir()), 'Unknown files in cache directory; use a new ROOT.'
    identity.write_text(json.dumps(manifest))
need = sum(manifest['sizes'][n] for n in names if not (CACHE/n).exists())
assert shutil.disk_usage(CACHE).free > need + 4*2**30, 'Need enough disk for cache plus 4 GiB.'
for name in names:
    dest=CACHE/name
    if dest.exists():
        assert dest.stat().st_size == manifest['sizes'][name]
        continue
    print('Copying',name,flush=True)
    temp=dest.with_name(dest.name+'.partial')
    shutil.copyfile(SOURCE/name,temp)
    assert temp.stat().st_size == manifest['sizes'][name]
    os.replace(temp,dest)
print('Local cache ready.')


In [ ]:
searcher=s.TestSearcher(CACHE,256,200)
faiss.omp_set_num_threads(4)
assert name_config['cache_rows']==searcher.total
missing_pool=pd.read_pickle(NAME_RUN/'name_pool/records.pkl')
assert missing_pool.rid.is_unique and missing_pool.entity_id.is_unique
assert missing_pool.business_address.fillna('').map(lambda x:not str(x).strip()).all()
# Saved ordering is the vector-row mapping; validate part shapes before concatenation.
parts=[]
for start in range(0,len(missing_pool),5000):
    part=np.load(NAME_RUN/'name_pool/embedding_parts'/f'{start:09d}.npy')
    assert part.shape==(min(5000,len(missing_pool)-start),c.DIM)
    parts.append(part)
name_vectors=np.concatenate(parts).astype(np.float32);del parts
assert np.isfinite(name_vectors).all()
faiss.normalize_L2(name_vectors)
# Verify saved rid mapping on a random subset against the current combined cache.
check=missing_pool.sample(n=min(100,len(missing_pool)),random_state=41)
check_records=searcher.fetch(check.rid.to_numpy())
assert all(check_records[int(r.rid)][0]==r.entity_id for r in check.itertuples())
name_indexes={}
for country in sorted(missing_pool.country.map(c.norm).unique()):
    ix=np.flatnonzero(missing_pool.country.map(c.norm).to_numpy()==country)
    idx=faiss.IndexIDMap2(faiss.IndexFlatIP(c.DIM))
    idx.add_with_ids(np.ascontiguousarray(name_vectors[ix]),np.ascontiguousarray(missing_pool.rid.to_numpy()[ix],dtype=np.int64))
    name_indexes[country]=idx
print('Reused blank-address name embeddings:',len(missing_pool))


In [ ]:
# Persistent name-vector bank avoids re-encoding repeated candidate names.
BANK=sqlite3.connect(ROOT/'name_vectors.sqlite')
BANK.execute('CREATE TABLE IF NOT EXISTS vectors (name TEXT PRIMARY KEY, vector BLOB NOT NULL)')
BANK.execute('PRAGMA cache_size=-65536')
seed_done=ROOT/'name_bank_seeded.json'
if not seed_done.exists():
    for start in range(0,len(missing_pool),5000):
        rows=[(c.clean_embedding_text(str(missing_pool.iloc[j].business_name)),sqlite3.Binary(name_vectors[j].tobytes()))
              for j in range(start,min(start+5000,len(missing_pool)))]
        with BANK:BANK.executemany('INSERT OR IGNORE INTO vectors VALUES (?,?)',rows)
    c.atomic_json(seed_done,{'rows':len(missing_pool)})
del name_vectors;gc.collect()
encoder=None

def encode_names(names):
    global encoder
    if encoder is None:encoder=c.Encoder(BATCH_SIZE)
    result=[]
    with torch.inference_mode():
        for start in range(0,len(names),BATCH_SIZE):
            texts=['query: '+n for n in names[start:start+BATCH_SIZE]]
            inputs=encoder.tokenizer(texts,max_length=256,padding=True,truncation=True,return_tensors='pt').to('cuda')
            hidden=encoder.model(**inputs).last_hidden_state.float()
            mask=inputs['attention_mask'].unsqueeze(-1)
            vec=(hidden*mask).sum(1)/mask.sum(1).clamp(min=1)
            vec=torch.nn.functional.normalize(vec,p=2,dim=1)
            result.append(vec.cpu().numpy())
    return np.concatenate(result).astype(np.float32) if result else np.empty((0,c.DIM),np.float32)

def names_from_bank(raw_names):
    keys=list(dict.fromkeys(c.clean_embedding_text(str(n)) for n in raw_names))
    found={}
    for start in range(0,len(keys),700):
        chunk=keys[start:start+700]
        sql='SELECT name,vector FROM vectors WHERE name IN ('+','.join('?'*len(chunk))+')'
        for name,blob in BANK.execute(sql,chunk):found[name]=np.frombuffer(blob,dtype=np.float32).copy()
    absent=[k for k in keys if k not in found]
    for start in range(0,len(absent),5000):
        chunk=absent[start:start+5000];vectors=encode_names(chunk)
        with BANK:
            BANK.executemany('INSERT OR IGNORE INTO vectors VALUES (?,?)',[(n,sqlite3.Binary(v.tobytes())) for n,v in zip(chunk,vectors)])
        found.update(zip(chunk,vectors))
    return found

query_names_path=ROOT/'query_names.npy'
if query_names_path.exists():query_names=np.load(query_names_path)
else:
    result=[]
    for start in range(0,len(sample),1000):
        names=sample.business_name.iloc[start:start+1000].tolist()
        bank=names_from_bank(names)
        result.append(np.stack([bank[c.clean_embedding_text(str(n))] for n in names]))
        print('Query name embeddings:',min(start+1000,len(sample)),'/',len(sample),flush=True)
    query_names=np.concatenate(result)
    with query_names_path.with_suffix('.partial').open('wb') as f:np.save(f,query_names)
    os.replace(query_names_path.with_suffix('.partial'),query_names_path)
assert query_names.shape==(len(sample),c.DIM)


In [ ]:
# Generate identical candidate unions for all matching variants.
# Ground truth enters only when labels are assigned AFTER retrieval.
PAIR_DIR=ROOT/'pairs';PAIR_DIR.mkdir(exist_ok=True)
started=time.perf_counter()
with threadpool_limits(limits=4):
    for start in range(0,len(sample),PAIR_CHUNK):
        owners=np.arange(start,min(start+PAIR_CHUNK,len(sample)))
        dest=PAIR_DIR/f'{start:07d}.npz'
        if dest.exists():
            with np.load(dest) as z:assert np.array_equal(z['query_indices'],owners)
            continue
        frame=sample.iloc[owners].reset_index(drop=True)
        base_ids,base_sim=searcher.search(frame,np.ascontiguousarray(q[owners],dtype=np.float32))
        name_ids=np.full((len(frame),5),-1,np.int64)
        countries=frame.country.map(c.norm).to_numpy()
        for country in np.unique(countries):
            loc=np.flatnonzero(countries==country)
            _,ids=name_indexes[country].search(np.ascontiguousarray(query_names[owners[loc]],dtype=np.float32),5)
            name_ids[loc]=ids
        unions=[list(dict.fromkeys(int(v) for v in list(base_ids[i])+list(name_ids[i]) if v>=0)) for i in range(len(frame))]
        records=searcher.fetch(np.array(sorted({rid for row in unions for rid in row}),np.int64))
        bank=names_from_bank([r[1] for r in records.values()])
        xs=[];ys=[];pair_owners=[];rids=[];baseline_flags=[]
        for i,left in enumerate(frame[c.COLS].itertuples(index=False,name=None)):
            owner=int(owners[i]);base_set=set(map(int,base_ids[i]))
            for rid in unions[i]:
                right=records[rid]
                assert c.norm(left[3])==c.norm(right[3])
                v=np.asarray(searcher.vectors[rid],np.float32)
                combined=float(v.dot(q[owner])/max(np.linalg.norm(v),1e-12))
                nv=bank[c.clean_embedding_text(str(right[1]))]
                name_cos=float(nv.dot(query_names[owner]))
                xs.append(c.features(left,right)+[combined,name_cos])
                ys.append(int(right[0] in truth[left[0]]))
                pair_owners.append(owner);rids.append(rid);baseline_flags.append(rid in base_set)
        arrays=dict(x=np.asarray(xs,np.float32),y=np.asarray(ys,np.uint8),owners=np.asarray(pair_owners,np.int32),
                    rids=np.asarray(rids,np.int64),baseline=np.asarray(baseline_flags,bool),query_indices=owners)
        assert arrays['x'].shape[1]==20 and np.isfinite(arrays['x']).all()
        assert len(xs)<=len(frame)*25
        with dest.with_suffix('.partial').open('wb') as f:np.savez_compressed(f,**arrays)
        os.replace(dest.with_suffix('.partial'),dest)
        print(f'Pairs {owners[-1]+1:,}/{len(sample):,}; elapsed {(time.perf_counter()-started)/60:.1f} min',flush=True)
        assert shutil.disk_usage(ROOT).free>1024**3,'Low disk; completed pair checkpoints are saved.'
searcher.close();BANK.close();name_indexes.clear()
if encoder is not None:del encoder
encoder=None;gc.collect();torch.cuda.empty_cache()


In [ ]:
chunks=[]
for start in range(0,len(sample),PAIR_CHUNK):
    with np.load(PAIR_DIR/f'{start:07d}.npz') as z:
        assert np.array_equal(z['query_indices'],np.arange(start,min(start+PAIR_CHUNK,len(sample))))
        chunks.append({k:z[k].copy() for k in ['x','y','owners','rids','baseline']})
pairs={k:np.concatenate([p[k] for p in chunks]) for k in chunks[0]};del chunks
splits=sample.split.to_numpy();counts=np.array([len(truth[sid]) for sid in sample.entity_id],np.int32)
train_mask=splits[pairs['owners']]=='train'
tune_ids=np.flatnonzero(splits=='tune');eval_ids=np.flatnonzero(splits=='evaluation')
assert not set(tune_ids)&set(eval_ids)
feature_names=c.MATCH_FEATURES+['name_embedding_cosine']
assert len(feature_names)==20 and c.MATCH_FEATURES[10]=='address_missing'
# This shows whether old-style baseline retrieval supplies enough blank-address examples.
audit=[]
for scope,mask in [('baseline_pairs',train_mask & pairs['baseline']),('union_pairs',train_mask)]:
    for blank in [False,True]:
        m=mask & (pairs['x'][:,10].astype(bool)==blank)
        audit.append(dict(scope=scope,address_missing=blank,pairs=int(m.sum()),positives=int(pairs['y'][m].sum()),negatives=int(m.sum()-pairs['y'][m].sum())))
pd.DataFrame(audit).to_csv(OUT/'training_pair_audit.csv',index=False)
display(pd.DataFrame(audit))
print('Total union training pairs:',int(train_mask.sum()))


In [ ]:
MODEL_DIR=ROOT/'models';MODEL_DIR.mkdir(exist_ok=True)
models={'saved_model':(old_bundle['model'],19)}
settings=[('baseline_pair_control19',19,train_mask & pairs['baseline']),
          ('expanded_pairs19',19,train_mask),('expanded_pairs20',20,train_mask)]
with threadpool_limits(limits=4):
    for name,nfeatures,mask in settings:
        path=MODEL_DIR/f'{name}.pkl'
        if path.exists():
            with path.open('rb') as f:model=pickle.load(f)
        else:
            print('Training',name,'on',int(mask.sum()),'pairs',flush=True)
            model=HistGradientBoostingClassifier(learning_rate=.08,max_iter=250,max_leaf_nodes=31,
                l2_regularization=1,early_stopping=False,random_state=2026)
            model.fit(pairs['x'][mask,:nfeatures],pairs['y'][mask])
            with path.with_suffix('.partial').open('wb') as f:pickle.dump(model,f)
            os.replace(path.with_suffix('.partial'),path)
        assert model.n_features_in_==nfeatures
        models[name]=(model,nfeatures)


In [ ]:
def metrics(prob,selected,threshold,include=None):
    if include is None:include=np.ones(len(prob),bool)
    owners=pairs['owners'];truth_label=pairs['y'].astype(bool)
    chosen=include & (prob>=threshold)
    predicted=np.bincount(owners[chosen],minlength=len(sample))
    tp=np.bincount(owners[chosen & truth_label],minlength=len(sample))
    fp=predicted-tp;fn=counts-tp
    denom=1.25*tp+fp+.25*fn
    score=np.divide(1.25*tp,denom,out=np.zeros(len(sample)),where=denom!=0)
    score[(counts==0)&(predicted==0)]=1
    hits=np.bincount(owners[include & truth_label],minlength=len(sample))
    blank=pairs['x'][:,10].astype(bool)
    within=np.isin(owners,selected)
    return dict(macro_F05=float(score[selected].mean()),true_accepted=int(tp[selected].sum()),
        false_accepted=int(fp[selected].sum()),missed_search=int((counts[selected]-hits[selected]).sum()),
        retrieved_rejected=int((hits[selected]-tp[selected]).sum()),
        blank_true_retrieved=int((include & within & blank & truth_label).sum()),
        blank_true_accepted=int((chosen & within & blank & truth_label).sum()),
        blank_false_accepted=int((chosen & within & blank & ~truth_label).sum()),
        singletons=int((counts[selected]==0).sum()),
        singletons_correct=int(((counts[selected]==0)&(predicted[selected]==0)).sum()))

# Tune and select before looking at evaluation labels/scores.
held_mask=splits[pairs['owners']]!='train'
probabilities={};tuning=[];chosen_thresholds={}
with threadpool_limits(limits=4):
    for name,(model,nfeatures) in models.items():
        prob=np.full(len(pairs['y']),np.nan,np.float32)
        prob[held_mask]=model.predict_proba(pairs['x'][held_mask,:nfeatures])[:,1]
        probabilities[name]=prob
        trials=[dict(model=name,threshold=float(t),**metrics(prob,tune_ids,float(t))) for t in np.linspace(.01,.99,99)]
        best=max(trials,key=lambda r:(r['macro_F05'],r['threshold']))
        tuning.extend(trials);chosen_thresholds[name]=best['threshold']
        print('Tuning best:',best,flush=True)
selected_model=max(models,key=lambda name:(max(r['macro_F05'] for r in tuning if r['model']==name),name))
c.atomic_json(OUT/'selection_before_evaluation.json',dict(selected_model=selected_model,thresholds=chosen_thresholds,
    note='Selected by tuning macro F0.5 only. Existing evaluation split was used in prior experiments.'))
pd.DataFrame(tuning).to_csv(OUT/'threshold_tuning.csv',index=False)

fixed_threshold=float(json.loads((UPGRADE/'upgrade_report.json').read_text())['tuning_options']['standard']['threshold'])
results=[]
for split,selected in [('tune',tune_ids),('evaluation',eval_ids)]:
    old_prob=probabilities['saved_model']
    results.append(dict(split=split,model='saved_baseline20_fixed',threshold=fixed_threshold,
                        **metrics(old_prob,selected,fixed_threshold,pairs['baseline'])))
    results.append(dict(split=split,model='saved_union25_fixed',threshold=fixed_threshold,
                        **metrics(old_prob,selected,fixed_threshold)))
    for name in models:
        t=chosen_thresholds[name]
        results.append(dict(split=split,model=name+'_tuned',threshold=t,**metrics(probabilities[name],selected,t)))
pd.DataFrame(results).to_csv(OUT/'model_comparison.csv',index=False)
display(pd.DataFrame(results))
print('Model selected on tuning:',selected_model)

# Save feature order and retrieval recipe alongside each experimental model.
for name,(model,nfeatures) in models.items():
    artifact={'model':model,'threshold':chosen_thresholds[name],'features':feature_names[:nfeatures],
              'retrieval':{'nprobe':256,'overfetch':200,'baseline_k':20,'blank_address_name_k':5},
              'version':'matcher-name-rescue-experimental-v1'}
    with (MODEL_DIR/f'{name}_bundle.pkl').open('wb') as f:pickle.dump(artifact,f)
metadata={'selected_on_tuning':selected_model,'features':feature_names,'config':config,
          'warning':'New 20-feature bundles are NOT compatible with the old 19-feature submission runner. No deployment was performed.'}
c.atomic_json(OUT/'experiment_metadata.json',metadata)


In [ ]:
# Preserve pair-level held-out scores for error analysis without re-running retrieval.
held=np.flatnonzero(held_mask)
score_table=pd.DataFrame({'source1_id':sample.entity_id.to_numpy()[pairs['owners'][held]],
    'split':splits[pairs['owners'][held]],'candidate_rid':pairs['rids'][held],
    'label':pairs['y'][held],'in_baseline20':pairs['baseline'][held],
    'address_missing':pairs['x'][held,10].astype(bool),
    'combined_cosine':pairs['x'][held,18],'name_cosine':pairs['x'][held,19]})
for name,prob in probabilities.items():score_table[name+'_score']=prob[held]
score_table.to_csv(OUT/'heldout_pair_scores.tsv',sep='\t',index=False)
shutil.copy2(cp,OUT/'configuration.json')
archive=ROOT/'matcher_name_rescue_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()):z.write(path,path.name)
print('Download results:',archive)
print('Retain full saved notebook output: models, pairs, query_names.npy and name_vectors.sqlite.')
print('Do not substitute these model files into the old submission runner; feature and candidate generation compatibility must be implemented first.')
